# Session 12 · Evaluating Classifiers

**Machine Learning Foundations · Sanketana School of Code**

Every classifier so far was graded on **accuracy**, and it behaved because our classes were balanced. Today we break it — on purpose — with **credit-card fraud**, where only ~4% of transactions are fraud. We're about to meet a model that scores **96% accuracy** and catches **zero fraud**.

By the end of this notebook you will be able to:

- explain **why accuracy misleads** when one class is rare
- name the two mistakes: **false positive** (false alarm) and **false negative** (a miss)
- compute **precision** and **recall**, and say what each one means
- make the ethics call: **which mistake is worse, and who pays?**

## Warm-up · Last session's homework

Your coach will walk through Session 11's threshold table and multi-class model (about 10 minutes). Moving the cut traded *false passes* for *missed passers* — two mistakes we never named. Today they get names.

## Step 1 · Meet the imbalance

Load the fraud data and look at how rare fraud is. **Predict first:** if you built a model that just guesses *legit* every time, what accuracy would it get?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
print("transactions:", len(fraud))
print("fraud rate:", round(fraud["is_fraud"].mean(), 3), " <-- only ~4% is fraud")
fraud.head()

In [ ]:
X = fraud.drop(columns=["is_fraud", "transaction_id"]).values
y = fraud["is_fraud"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("test transactions:", len(y_test), " | real frauds in test:", int(y_test.sum()))

## Step 2 · Watch accuracy lie

Build the laziest possible "model" — always predict **legit** — and score its accuracy. Then ask the question accuracy can't answer: **how much fraud did it catch?**

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
base_pred = baseline.predict(X_test)

print("always-legit accuracy:", round(accuracy_score(y_test, base_pred), 3), " <-- looks like an A")
frauds_caught = int(((base_pred == 1) & (y_test == 1)).sum())
print("frauds it caught:     ", frauds_caught, "/", int(y_test.sum()), " <-- caught NOTHING")

**Sit with that.** A model that is right **96%** of the time caught **0** of the frauds it exists to stop. High accuracy hid total failure — because the rare class is the one that matters, and accuracy is dominated by the common one.

## Step 3 · Name the two mistakes

There are only four things that can happen on each transaction:

```
                     MODEL SAYS
                fraud            legit
 REALLY fraud   caught  ✓        MISSED   ✗   <- false negative: a thief gets through
        legit   false alarm ✗    fine     ✓   <- false positive: an honest card frozen
```

- **False positive (false alarm):** flag an honest transaction — an innocent customer's card is frozen.
- **False negative (a miss):** wave a real fraud through — someone gets robbed.

Accuracy lumps all of these together. Precision and recall pull them apart.

## Step 4 · Precision and recall — the metrics that see the truth

Train a **real** logistic model on the same data, and compute three numbers:

- **recall** = of all the real fraud, how much did we catch?
- **precision** = when it cried fraud, how often was it right?

In [ ]:
# ✏️ TODO: fit logistic regression on the scaled training data.
model = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
pred = model.predict(X_test_s)

acc = accuracy_score(y_test, pred)
prec = precision_score(y_test, pred)
rec = recall_score(y_test, pred)
print("real model  accuracy :", round(acc, 3))
print("real model  precision:", round(prec, 2), " (of its fraud alarms, this fraction was real)")
print("real model  recall   :", round(rec, 2), " (of all real fraud, this fraction was caught)")
caught = int(((pred == 1) & (y_test == 1)).sum())
print("frauds caught:", caught, "/", int(y_test.sum()))

In [ ]:
# The whole point, in one chart: accuracy barely moves, recall tells the real story.
labels = ["accuracy", "recall"]
base_vals = [accuracy_score(y_test, base_pred), recall_score(y_test, base_pred, zero_division=0)]
real_vals = [acc, rec]
x = np.arange(2); w = 0.35
plt.figure(figsize=(7, 4))
plt.bar(x - w/2, base_vals, w, label="always-legit baseline")
plt.bar(x + w/2, real_vals, w, label="real logistic model")
plt.xticks(x, labels); plt.ylim(0, 1.05); plt.ylabel("score")
plt.title("Accuracy can't tell the models apart — recall can")
plt.legend(); plt.tight_layout(); plt.show()

**Read it.** On **accuracy** the two models look almost identical (0.96 vs 0.98). On **recall** they're worlds apart (0.00 vs ~0.60). The metric you choose decides whether you can even *see* the difference between useless and useful.

## Step 5 · ✏️ The ethics call — which mistake is worse?

This is the heart of the session, and there is **no answer key**. For a bank running this fraud detector:

- a **false negative** (miss real fraud) means the bank and a customer lose money to a thief;
- a **false positive** (freeze an honest card) means a real person is stranded, unable to pay, and furious.

**Write 3–4 sentences:** which mistake do *you* think is worse for this bank and its customers, and — crucially — **who bears the cost** of each one? Defend your choice. (A good answer names the people who pay, not just a preference.)

*Your answer here:*

## Wrap-up

- On **imbalanced** data, **accuracy hides failure** — the 96%/0-fraud baseline is the proof.
- The two mistakes: **false positive** (false alarm) and **false negative** (a miss).
- **Recall** = of the real fraud, how much we caught; **precision** = of our alarms, how many were real.
- Which mistake is *worse* is a **human judgement about who pays** — a number can't decide it.

**Next session:** we lay all four outcomes in one grid — the **confusion matrix** — and slide the threshold to watch **precision and recall trade off**: catch more fraud, annoy more honest customers.

*Homework: make the accuracy trap undeniable and take a stand, in `homework.ipynb`.*